# 10 Final Test Evaluation

## 10.0 Purpose and fixed rules

This notebook performs the final chronological test evaluation for the B2C email engagement prediction thesis.

Stage 9 refitted the selected tuned models on the full train-validation set and saved the fitted sklearn pipelines. Stage 10 loads those frozen fitted pipelines, recreates the same chronological final test split, evaluates the final test set once, and saves all final-test outputs.

## Fixed rules

1. This notebook is evaluation-only.
2. No model tuning is allowed.
3. No feature engineering changes are allowed.
4. No feature set changes are allowed.
5. No threshold tuning on the final test set is allowed.
6. The final test set must not be used to revise model choices.
7. All saved Stage 9 pipelines must be treated as frozen.
8. Final test predictions and metrics are computed once and documented.
9. Open prediction uses threshold 0.5 for the confusion matrix.
10. Click prediction is evaluated mainly through PR-AUC and top-k/lift analysis.
11. Click accuracy and threshold-0.5 confusion matrix are contextual only.
12. Any performance drop from CV to final test must be documented, not “fixed” by retuning.

## 10.1 Imports and paths

In [1]:
# imports
import json
import platform
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)

In [5]:
# constants
RANDOM_STATE = 42
N_JOBS = 1

GROUP_COL = "mailing_id"
USER_COL = "user_id"

OPEN_TARGET = "open"
CLICK_TARGET = "click"

OPEN_THRESHOLD = 0.5
CLICK_CONTEXTUAL_THRESHOLD = 0.5

TOPK_FRACTIONS = [0.001, 0.005, 0.01, 0.05, 0.10]       # 0.1%, 0.5%, 1%, 5%, 10%

ALLOW_OVERWRITE_FINAL_TEST_OUTPUTS = False

In [6]:
# file paths
THESIS_DIR = Path.home() / "Documents" / "Thesis"

DATA_DIR = THESIS_DIR / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed Data"
OUTPUT_DIR = THESIS_DIR / "Outputs"

OPEN_MODEL_PATH = PROCESSED_DATA_DIR / "df_open_model_v1.parquet"
CLICK_MODEL_PATH = PROCESSED_DATA_DIR / "df_click_model_v1.parquet"

STAGE9_METADATA_PATH = OUTPUT_DIR / "9_refit_metadata_v1.json"
STAGE9_FEATURE_NAMES_PATH = OUTPUT_DIR / "9_refit_fitted_feature_names_v1.json"
STAGE9_SUMMARY_PATH = OUTPUT_DIR / "9_refit_summary_v1.csv"

OPEN_TUNING_SUMMARY_PATH = OUTPUT_DIR / "7_open_tuned_model_summary_v1.csv"
CLICK_TUNING_SUMMARY_PATH = OUTPUT_DIR / "8_click_tuned_model_summary_v1.csv"

print("Paths defined.")
print("Thesis directory:", THESIS_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Paths defined.
Thesis directory: /Users/khanhhien/Documents/Thesis
Processed data directory: /Users/khanhhien/Documents/Thesis/Data/Processed Data
Output directory: /Users/khanhhien/Documents/Thesis/Outputs


## 10.2 Load Stage 9 metadata, feature names, and refit summary

In [7]:
with open(STAGE9_METADATA_PATH, "r") as f:
    stage9_metadata = json.load(f)

with open(STAGE9_FEATURE_NAMES_PATH, "r") as f:
    stage9_fitted_feature_names = json.load(f)

stage9_refit_summary = pd.read_csv(STAGE9_SUMMARY_PATH)

print("Stage 9 metadata loaded.")
print("Stage 9 fitted feature names loaded.")
print("Stage 9 refit summary loaded.")

print("\nRefit summary shape:", stage9_refit_summary.shape)

Stage 9 metadata loaded.
Stage 9 fitted feature names loaded.
Stage 9 refit summary loaded.

Refit summary shape: (5, 20)


In [8]:
# inspect
print("Metadata keys:")
print(stage9_metadata.keys())

print("\nModel paths:")
for key, path in stage9_metadata["model_paths"].items():
    print(key, "->", path)

print("\nFitted feature name keys:")
for key in stage9_fitted_feature_names.keys():
    print(key, ":", len(stage9_fitted_feature_names[key]), "features")

print("\nRefit summary columns:")
print(stage9_refit_summary.columns.tolist())

Metadata keys:
dict_keys(['project', 'stage', 'notebook', 'random_state', 'n_jobs', 'group_col', 'open_target', 'click_target', 'split_logic', 'final_test_status', 'open_split', 'click_split', 'feature_sets', 'feature_types', 'carry_forward_params', 'model_paths', 'summary_file', 'notes'])

Model paths:
open__logistic_regression__C_expanded -> /Users/khanhhien/Documents/Thesis/Outputs/9_open_refit_model_logistic_C_v1.joblib
open__decision_tree__C_expanded -> /Users/khanhhien/Documents/Thesis/Outputs/9_open_refit_model_decision_tree_C_v1.joblib
open__random_forest__C_expanded -> /Users/khanhhien/Documents/Thesis/Outputs/9_open_refit_model_random_forest_C_v1.joblib
click__logistic_regression__C_expanded -> /Users/khanhhien/Documents/Thesis/Outputs/9_click_refit_model_logistic_C_v1.joblib
click__random_forest__C_expanded -> /Users/khanhhien/Documents/Thesis/Outputs/9_click_refit_model_random_forest_C_v1.joblib

Fitted feature name keys:
open__logistic_regression__C_expanded : 60 features


## 10.3 Load saved fitted pipelines

In [9]:
# load models
models = {}

for model_key, model_path_str in stage9_metadata["model_paths"].items():
    
    model_path = Path(model_path_str)
    
    assert model_path.exists(), f"Missing model file: {model_path}"
    
    models[model_key] = joblib.load(model_path)
    
    print(model_key, "loaded.")

open__logistic_regression__C_expanded loaded.
open__decision_tree__C_expanded loaded.
open__random_forest__C_expanded loaded.
click__logistic_regression__C_expanded loaded.
click__random_forest__C_expanded loaded.


In [10]:
# verify
expected_model_keys = {
    "open__logistic_regression__C_expanded",
    "open__decision_tree__C_expanded",
    "open__random_forest__C_expanded",
    "click__logistic_regression__C_expanded",
    "click__random_forest__C_expanded",
}

actual_model_keys = set(models.keys())

print("Expected keys match:", actual_model_keys == expected_model_keys)

print("\nLoaded model keys:")
for key in sorted(actual_model_keys):
    print(key)

assert actual_model_keys == expected_model_keys, "Loaded model keys do not match expected model keys."

Expected keys match: True

Loaded model keys:
click__logistic_regression__C_expanded
click__random_forest__C_expanded
open__decision_tree__C_expanded
open__logistic_regression__C_expanded
open__random_forest__C_expanded


In [12]:
# verify pipeline structure
for model_key, pipeline in models.items():
    
    print("\n", model_key)
    print("Pipeline type:", type(pipeline).__name__)
    print("Steps:", list(pipeline.named_steps.keys()))
    
    assert "preprocessor" in pipeline.named_steps
    assert "model" in pipeline.named_steps


 open__logistic_regression__C_expanded
Pipeline type: Pipeline
Steps: ['preprocessor', 'model']

 open__decision_tree__C_expanded
Pipeline type: Pipeline
Steps: ['preprocessor', 'model']

 open__random_forest__C_expanded
Pipeline type: Pipeline
Steps: ['preprocessor', 'model']

 click__logistic_regression__C_expanded
Pipeline type: Pipeline
Steps: ['preprocessor', 'model']

 click__random_forest__C_expanded
Pipeline type: Pipeline
Steps: ['preprocessor', 'model']


## 10.4 Load open/click modelling datasets

In [13]:
# load datasets
df_open = pd.read_parquet(OPEN_MODEL_PATH)
df_click = pd.read_parquet(CLICK_MODEL_PATH)

print("Open dataset shape:", df_open.shape)
print("Click dataset shape:", df_click.shape)

Open dataset shape: (1057544, 38)
Click dataset shape: (1104242, 38)


In [14]:
# checks
for col in [USER_COL, GROUP_COL, OPEN_TARGET]:
    assert col in df_open.columns, f"Missing open column: {col}"

for col in [USER_COL, GROUP_COL, CLICK_TARGET]:
    assert col in df_click.columns, f"Missing click column: {col}"

assert df_open[OPEN_TARGET].isna().sum() == 0
assert df_click[CLICK_TARGET].isna().sum() == 0

print("Open target distribution:")
print(df_open[OPEN_TARGET].value_counts(normalize=True).sort_index())

print("\nClick target distribution:")
print(df_click[CLICK_TARGET].value_counts(normalize=True).sort_index())

print("\nOpen unique campaigns:", df_open[GROUP_COL].nunique())
print("Click unique campaigns:", df_click[GROUP_COL].nunique())

Open target distribution:
open
0.0    0.46382
1.0    0.53618
Name: proportion, dtype: float64

Click target distribution:
click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

Open unique campaigns: 274
Click unique campaigns: 274


## 10.5 Recreate chronological trainval/test split

In [15]:
# campaign split
campaign_ids = sorted(df_open[GROUP_COL].unique())

n_campaigns = len(campaign_ids)
n_trainval_campaigns = int(n_campaigns * 0.80)

trainval_campaign_ids = campaign_ids[:n_trainval_campaigns]
test_campaign_ids = campaign_ids[n_trainval_campaigns:]

print("Total campaigns:", n_campaigns)
print("Trainval campaigns:", len(trainval_campaign_ids))
print("Test campaigns:", len(test_campaign_ids))

print("\nTrainval campaign range:", min(trainval_campaign_ids), "to", max(trainval_campaign_ids))
print("Test campaign range:", min(test_campaign_ids), "to", max(test_campaign_ids))

campaign_overlap = set(trainval_campaign_ids).intersection(test_campaign_ids)

print("\nCampaign overlap:", len(campaign_overlap))

assert len(campaign_overlap) == 0

Total campaigns: 274
Trainval campaigns: 219
Test campaigns: 55

Trainval campaign range: 653 to 1252
Test campaign range: 1257 to 1418

Campaign overlap: 0


In [16]:
# create split datasets
open_trainval_df = df_open[df_open[GROUP_COL].isin(trainval_campaign_ids)].copy()
open_test_df = df_open[df_open[GROUP_COL].isin(test_campaign_ids)].copy()

click_trainval_df = df_click[df_click[GROUP_COL].isin(trainval_campaign_ids)].copy()
click_test_df = df_click[df_click[GROUP_COL].isin(test_campaign_ids)].copy()

print("Open trainval:", open_trainval_df.shape)
print("Open test:", open_test_df.shape)

print("\nClick trainval:", click_trainval_df.shape)
print("Click test:", click_test_df.shape)

Open trainval: (781826, 38)
Open test: (275718, 38)

Click trainval: (814269, 38)
Click test: (289973, 38)


## 10.6 Verify split, feature columns, and class ordering

In [17]:
# split verification
expected_open_split = stage9_metadata["open_split"]
expected_click_split = stage9_metadata["click_split"]

split_checks = {
    "open_trainval_rows": (
        len(open_trainval_df),
        expected_open_split["trainval_rows"]
    ),
    "open_test_rows": (
        len(open_test_df),
        expected_open_split["test_rows"]
    ),
    "open_trainval_campaigns": (
        open_trainval_df[GROUP_COL].nunique(),
        expected_open_split["trainval_campaigns"]
    ),
    "open_test_campaigns": (
        open_test_df[GROUP_COL].nunique(),
        expected_open_split["test_campaigns"]
    ),
    "open_trainval_campaign_min": (
        int(open_trainval_df[GROUP_COL].min()),
        expected_open_split["trainval_campaign_min"]
    ),
    "open_trainval_campaign_max": (
        int(open_trainval_df[GROUP_COL].max()),
        expected_open_split["trainval_campaign_max"]
    ),
    "open_test_campaign_min": (
        int(open_test_df[GROUP_COL].min()),
        expected_open_split["test_campaign_min"]
    ),
    "open_test_campaign_max": (
        int(open_test_df[GROUP_COL].max()),
        expected_open_split["test_campaign_max"]
    ),
    "click_trainval_rows": (
        len(click_trainval_df),
        expected_click_split["trainval_rows"]
    ),
    "click_test_rows": (
        len(click_test_df),
        expected_click_split["test_rows"]
    ),
    "click_trainval_campaigns": (
        click_trainval_df[GROUP_COL].nunique(),
        expected_click_split["trainval_campaigns"]
    ),
    "click_test_campaigns": (
        click_test_df[GROUP_COL].nunique(),
        expected_click_split["test_campaigns"]
    ),
    "click_trainval_campaign_min": (
        int(click_trainval_df[GROUP_COL].min()),
        expected_click_split["trainval_campaign_min"]
    ),
    "click_trainval_campaign_max": (
        int(click_trainval_df[GROUP_COL].max()),
        expected_click_split["trainval_campaign_max"]
    ),
    "click_test_campaign_min": (
        int(click_test_df[GROUP_COL].min()),
        expected_click_split["test_campaign_min"]
    ),
    "click_test_campaign_max": (
        int(click_test_df[GROUP_COL].max()),
        expected_click_split["test_campaign_max"]
    ),
}

for check_name, (actual, expected) in split_checks.items():
    print(f"{check_name}: actual={actual}, expected={expected}")
    assert actual == expected, f"{check_name} mismatch: actual={actual}, expected={expected}"

open_overlap = set(open_trainval_df[GROUP_COL]).intersection(set(open_test_df[GROUP_COL]))
click_overlap = set(click_trainval_df[GROUP_COL]).intersection(set(click_test_df[GROUP_COL]))

print("\nOpen campaign overlap:", len(open_overlap))
print("Click campaign overlap:", len(click_overlap))

assert len(open_overlap) == 0
assert len(click_overlap) == 0

open_trainval_rows: actual=781826, expected=781826
open_test_rows: actual=275718, expected=275718
open_trainval_campaigns: actual=219, expected=219
open_test_campaigns: actual=55, expected=55
open_trainval_campaign_min: actual=653, expected=653
open_trainval_campaign_max: actual=1252, expected=1252
open_test_campaign_min: actual=1257, expected=1257
open_test_campaign_max: actual=1418, expected=1418
click_trainval_rows: actual=814269, expected=814269
click_test_rows: actual=289973, expected=289973
click_trainval_campaigns: actual=219, expected=219
click_test_campaigns: actual=55, expected=55
click_trainval_campaign_min: actual=653, expected=653
click_trainval_campaign_max: actual=1252, expected=1252
click_test_campaign_min: actual=1257, expected=1257
click_test_campaign_max: actual=1418, expected=1418

Open campaign overlap: 0
Click campaign overlap: 0


In [18]:
# feature verification
open_features_expected = stage9_metadata["feature_sets"]["open_C_expanded"]
click_features_expected = stage9_metadata["feature_sets"]["click_C_expanded"]

print("Expected open features:", len(open_features_expected))
print("Expected click features:", len(click_features_expected))

missing_open_features = [
    col for col in open_features_expected
    if col not in open_trainval_df.columns
]

missing_click_features = [
    col for col in click_features_expected
    if col not in click_trainval_df.columns
]

print("\nMissing open features:")
print(missing_open_features)

print("\nMissing click features:")
print(missing_click_features)

assert len(missing_open_features) == 0
assert len(missing_click_features) == 0

Expected open features: 28
Expected click features: 28

Missing open features:
[]

Missing click features:
[]


In [19]:
# class ordering verification
positive_class_index = {}

for model_key, pipeline in models.items():

    classes = (
        pipeline
        .named_steps["model"]
        .classes_
        .tolist()
    )

    print("\n", model_key)
    print("Classes:", classes)

    assert 1.0 in classes, f"Class 1 missing for {model_key}"

    positive_class_index[model_key] = classes.index(1.0)

    print(
        "Positive class index:",
        positive_class_index[model_key]
    )

print("\nSummary")

for model_key, idx in positive_class_index.items():
    print(model_key, "->", idx)


 open__logistic_regression__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

 open__decision_tree__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

 open__random_forest__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

 click__logistic_regression__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

 click__random_forest__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

Summary
open__logistic_regression__C_expanded -> 1
open__decision_tree__C_expanded -> 1
open__random_forest__C_expanded -> 1
click__logistic_regression__C_expanded -> 1
click__random_forest__C_expanded -> 1


## 10.7 Final-test preflight lock

In [20]:
preflight_checks = {
    "stage9_metadata_loaded": "stage9_metadata" in globals(),
    "stage9_feature_names_loaded": "stage9_fitted_feature_names" in globals(),
    "stage9_refit_summary_loaded": "stage9_refit_summary" in globals(),
    "models_loaded": "models" in globals() and len(models) == 5,
    "open_dataset_loaded": "df_open" in globals(),
    "click_dataset_loaded": "df_click" in globals(),
    "open_split_created": "open_trainval_df" in globals() and "open_test_df" in globals(),
    "click_split_created": "click_trainval_df" in globals() and "click_test_df" in globals(),
    "positive_class_index_checked": "positive_class_index" in globals() and len(positive_class_index) == 5,
    "open_features_available": all(col in open_test_df.columns for col in open_features_expected),
    "click_features_available": all(col in click_test_df.columns for col in click_features_expected),
    "click_current_open_excluded": OPEN_TARGET not in click_features_expected,
    "overwrite_protection_active": ALLOW_OVERWRITE_FINAL_TEST_OUTPUTS is False,
}

for check_name, passed in preflight_checks.items():
    print(f"{check_name}: {passed}")
    assert passed, f"Preflight check failed: {check_name}"

stage9_metadata_loaded: True
stage9_feature_names_loaded: True
stage9_refit_summary_loaded: True
models_loaded: True
open_dataset_loaded: True
click_dataset_loaded: True
open_split_created: True
click_split_created: True
positive_class_index_checked: True
open_features_available: True
click_features_available: True
click_current_open_excluded: True
overwrite_protection_active: True


## 10.8 Build final-test dummy baselines

In [21]:
open_trainval_rate = open_trainval_df[OPEN_TARGET].mean()
click_trainval_rate = click_trainval_df[CLICK_TARGET].mean()

open_majority_class = int(open_trainval_rate >= 0.5)
click_majority_class = int(click_trainval_rate >= 0.5)

print("Open trainval rate:", open_trainval_rate)
print("Open majority class:", open_majority_class)

print("\nClick trainval rate:", click_trainval_rate)
print("Click majority class:", click_majority_class)

Open trainval rate: 0.5732733370340716
Open majority class: 1

Click trainval rate: 0.001228095383712262
Click majority class: 0


In [22]:
open_baseline_probas = {
    "majority_class_baseline": np.full(
        len(open_test_df),
        open_majority_class,
        dtype=float
    ),
    "prior_probability_baseline": np.full(
        len(open_test_df),
        open_trainval_rate,
        dtype=float
    ),
}

click_baseline_probas = {
    "majority_class_baseline": np.full(
        len(click_test_df),
        click_majority_class,
        dtype=float
    ),
    "prior_probability_baseline": np.full(
        len(click_test_df),
        click_trainval_rate,
        dtype=float
    ),
}

print("Open baseline arrays:")
for name, arr in open_baseline_probas.items():
    print(name, arr.shape, "mean:", arr.mean())

print("\nClick baseline arrays:")
for name, arr in click_baseline_probas.items():
    print(name, arr.shape, "mean:", arr.mean())

Open baseline arrays:
majority_class_baseline (275718,) mean: 1.0
prior_probability_baseline (275718,) mean: 0.5732733370340716

Click baseline arrays:
majority_class_baseline (289973,) mean: 0.0
prior_probability_baseline (289973,) mean: 0.001228095383712262


##  10.9 Predict once on open final test

In [23]:
# prepare X/y
X_open_test = open_test_df[open_features_expected].copy()
y_open_test = open_test_df[OPEN_TARGET].copy()

print("X_open_test shape:", X_open_test.shape)
print("y_open_test shape:", y_open_test.shape)

print("\nOpen final test target distribution:")
print(y_open_test.value_counts(normalize=True).sort_index())

print("\nOpen final test positive count:", int(y_open_test.sum()))

X_open_test shape: (275718, 28)
y_open_test shape: (275718,)

Open final test target distribution:
open
0.0    0.569002
1.0    0.430998
Name: proportion, dtype: float64

Open final test positive count: 118834


In [30]:
# genertate open model probabilities
open_test_probas = {}

open_model_keys = [
    "open__logistic_regression__C_expanded",
    "open__decision_tree__C_expanded",
    "open__random_forest__C_expanded",
]

for model_key in open_model_keys:
    
    pipeline = models[model_key]
    pos_idx = positive_class_index[model_key]
    
    open_test_probas[model_key] = pipeline.predict_proba(X_open_test)[:, pos_idx]
    
    print(
        model_key,
        "->",
        open_test_probas[model_key].shape,
        "mean:",
        open_test_probas[model_key].mean(),
        "min:",
        open_test_probas[model_key].min(),
        "max:",
        open_test_probas[model_key].max()
    )

open__logistic_regression__C_expanded -> (275718,) mean: 0.523232953252348 min: 0.010709262373313466 max: 0.9411434195339329
open__decision_tree__C_expanded -> (275718,) mean: 0.6029176562743671 min: 0.10670297403893675 max: 0.9110644900108088
open__random_forest__C_expanded -> (275718,) mean: 0.5939197417840363 min: 0.047402255297586675 max: 0.9298748392637818


In [31]:
# open prediction dataframe
open_final_test_predictions = pd.DataFrame({
    USER_COL: open_test_df[USER_COL].values,
    GROUP_COL: open_test_df[GROUP_COL].values,
    "y_true": y_open_test.values,
    "proba_logistic_C": open_test_probas["open__logistic_regression__C_expanded"],
    "proba_decision_tree_C": open_test_probas["open__decision_tree__C_expanded"],
    "proba_random_forest_C": open_test_probas["open__random_forest__C_expanded"],
})

open_final_test_predictions["pred_label_logistic_C_at_0_5"] = (
    open_final_test_predictions["proba_logistic_C"] >= OPEN_THRESHOLD
).astype(int)

open_final_test_predictions["pred_label_decision_tree_C_at_0_5"] = (
    open_final_test_predictions["proba_decision_tree_C"] >= OPEN_THRESHOLD
).astype(int)

open_final_test_predictions["pred_label_random_forest_C_at_0_5"] = (
    open_final_test_predictions["proba_random_forest_C"] >= OPEN_THRESHOLD
).astype(int)

print("Open final test predictions shape:", open_final_test_predictions.shape)
open_final_test_predictions.head()

Open final test predictions shape: (275718, 9)


,user_id,mailing_id,y_true,proba_logistic_C,proba_decision_tree_C,proba_random_forest_C,pred_label_logistic_C_at_0_5,pred_label_decision_tree_C_at_0_5,pred_label_random_forest_C_at_0_5
0,5c6bebde5798a794283224c9,1274,0.0,0.514696,0.530022,0.549710,1,1,1
1,5c6bebde5798a794283224c9,1281,0.0,0.356221,0.530022,0.537686,0,1,1
2,5c6bebde5798a794283224c9,1288,0.0,0.365967,0.530022,0.526663,0,1,1
3,5c6bebde5798a794283224c9,1297,0.0,0.429640,0.530022,0.530246,0,1,1
4,5c6bebde5798a794283224c9,1301,0.0,0.349798,0.530022,0.521715,0,1,1


In [36]:
open_test_df.reset_index(drop=True).loc[
    open_final_test_predictions.head().index,
    [
        USER_COL,
        GROUP_COL,
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_rate",
        "prior_email_count",
        "historical_open_bucket",
    ]
]

,user_id,mailing_id,campaign_topic,subject_length_group,preheader_length_group,historical_open_rate,prior_email_count,historical_open_bucket
0,5c6bebde5798a794283224c9,1274,Automotive & Mobility,very_long,very_long,0.440000,50,moderate
1,5c6bebde5798a794283224c9,1281,Media & Publishing,long,medium,0.431373,51,moderate
2,5c6bebde5798a794283224c9,1288,Automotive & Mobility,medium,short,0.423077,52,moderate
3,5c6bebde5798a794283224c9,1297,Lottery & Games,long,very_long,0.415094,53,moderate
4,5c6bebde5798a794283224c9,1301,Automotive & Mobility,medium,short,0.407407,54,moderate


## 10.10 Predict once on click final test

In [26]:
# prepare X/y
X_click_test = click_test_df[click_features_expected].copy()
y_click_test = click_test_df[CLICK_TARGET].copy()

print("X_click_test shape:", X_click_test.shape)
print("y_click_test shape:", y_click_test.shape)

print("\nClick final test target distribution:")
print(y_click_test.value_counts(normalize=True).sort_index())

print("\nClick final test positive count:", int(y_click_test.sum()))

X_click_test shape: (289973, 28)
y_click_test shape: (289973,)

Click final test target distribution:
click
0.0    0.999328
1.0    0.000672
Name: proportion, dtype: float64

Click final test positive count: 195


In [32]:
# generate click model probabilities
click_test_probas = {}

click_model_keys = [
    "click__logistic_regression__C_expanded",
    "click__random_forest__C_expanded",
]

for model_key in click_model_keys:
    
    pipeline = models[model_key]
    pos_idx = positive_class_index[model_key]
    
    click_test_probas[model_key] = pipeline.predict_proba(X_click_test)[:, pos_idx]
    
    print(
        model_key,
        "->",
        click_test_probas[model_key].shape,
        "mean:",
        click_test_probas[model_key].mean(),
        "min:",
        click_test_probas[model_key].min(),
        "max:",
        click_test_probas[model_key].max()
    )

click__logistic_regression__C_expanded -> (289973,) mean: 0.0014667777194980211 min: 6.676293157482111e-06 max: 0.9206898559635099
click__random_forest__C_expanded -> (289973,) mean: 0.0016389078437987355 min: 6.372964725927241e-06 max: 0.812131884343726


In [33]:
# click prediction dataframe
click_final_test_predictions = pd.DataFrame({
    USER_COL: click_test_df[USER_COL].values,
    GROUP_COL: click_test_df[GROUP_COL].values,
    "y_true": y_click_test.values,
    "proba_logistic_C": click_test_probas["click__logistic_regression__C_expanded"],
    "proba_random_forest_C": click_test_probas["click__random_forest__C_expanded"],
})

click_final_test_predictions["pred_label_logistic_C_at_0_5"] = (
    click_final_test_predictions["proba_logistic_C"] >= CLICK_CONTEXTUAL_THRESHOLD
).astype(int)

click_final_test_predictions["pred_label_random_forest_C_at_0_5"] = (
    click_final_test_predictions["proba_random_forest_C"] >= CLICK_CONTEXTUAL_THRESHOLD
).astype(int)

print("Click final test predictions shape:", click_final_test_predictions.shape)
click_final_test_predictions.head()

Click final test predictions shape: (289973, 7)


,user_id,mailing_id,y_true,proba_logistic_C,proba_random_forest_C,pred_label_logistic_C_at_0_5,pred_label_random_forest_C_at_0_5
0,5c6bebde5798a794283224c9,1274,0.0,0.000347,0.001734,0,0
1,5c6bebde5798a794283224c9,1281,0.0,0.000264,0.000029,0,0
2,5c6bebde5798a794283224c9,1288,0.0,0.000921,0.000023,0,0
3,5c6bebde5798a794283224c9,1297,0.0,0.000226,0.001295,0,0
4,5c6bebde5798a794283224c9,1301,0.0,0.000883,0.000048,0,0


## 10.11 Compute open final test metrics

In [ ]:
# test metric function
def compute_binary_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    tn, fp, fn, tp = cm.ravel()
    
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "brier": brier_score_loss(y_true, y_proba),
        "log_loss": log_loss(y_true, y_proba, labels=[0, 1]),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "predicted_positive_count": int(y_pred.sum()),
        "predicted_positive_rate": float(y_pred.mean()),
        "proba_min": float(np.min(y_proba)),
        "proba_max": float(np.max(y_proba)),
        "proba_mean": float(np.mean(y_proba)),
        "proba_std": float(np.std(y_proba)),
        "proba_q01": float(np.quantile(y_proba, 0.01)),
        "proba_q05": float(np.quantile(y_proba, 0.05)),
        "proba_q50": float(np.quantile(y_proba, 0.50)),
        "proba_q95": float(np.quantile(y_proba, 0.95)),
        "proba_q99": float(np.quantile(y_proba, 0.99)),
    }

In [38]:
# compute open metrics
open_metric_rows = []

open_metric_inputs = {
    "majority_class_baseline": open_baseline_probas["majority_class_baseline"],
    "prior_probability_baseline": open_baseline_probas["prior_probability_baseline"],
    "logistic_regression__C_expanded": open_final_test_predictions["proba_logistic_C"].values,
    "decision_tree__C_expanded": open_final_test_predictions["proba_decision_tree_C"].values,
    "random_forest__C_expanded": open_final_test_predictions["proba_random_forest_C"].values,
}

for model_name, y_proba in open_metric_inputs.items():
    metrics = compute_binary_metrics(
        y_true=y_open_test,
        y_proba=y_proba,
        threshold=OPEN_THRESHOLD
    )
    
    row = {
        "target": "open",
        "model_name": model_name,
        "threshold": OPEN_THRESHOLD,
        "n_test_rows": len(y_open_test),
        "n_test_positive": int(y_open_test.sum()),
        "test_positive_rate": float(y_open_test.mean()),
        **metrics
    }
    
    open_metric_rows.append(row)

open_final_test_metrics = pd.DataFrame(open_metric_rows)

open_final_test_metrics

,target,model_name,threshold,n_test_rows,n_test_positive,test_positive_rate,roc_auc,pr_auc,accuracy,balanced_accuracy,...,predicted_positive_rate,proba_min,proba_max,proba_mean,proba_std,proba_q01,proba_q05,proba_q50,proba_q95,proba_q99
0,open,majority_class_baseline,0.5,275718,118834,0.430998,0.500000,0.430998,0.430998,0.500000,...,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,open,prior_probability_baseline,0.5,275718,118834,0.430998,0.500000,0.430998,0.430998,0.500000,...,1.000000,0.573273,0.573273,0.573273,0.000000,0.573273,0.573273,0.573273,0.573273,0.573273
2,open,logistic_regression__C_expanded,0.5,275718,118834,0.430998,0.773247,0.705224,0.689592,0.700113,...,0.548622,0.010709,0.941143,0.523233,0.260349,0.054651,0.106474,0.546033,0.868774,0.896657
3,open,decision_tree__C_expanded,0.5,275718,118834,0.430998,0.772588,0.670857,0.649914,0.675920,...,0.664168,0.106703,0.911064,0.602918,0.258098,0.106703,0.106703,0.705935,0.911064,0.911064
4,open,random_forest__C_expanded,0.5,275718,118834,0.430998,0.776073,0.715636,0.659990,0.683323,...,0.643777,0.047402,0.929875,0.593920,0.249238,0.107556,0.163457,0.626218,0.911409,0.919070


In [39]:
open_final_test_metrics[
    [
        "model_name",
        "roc_auc",
        "pr_auc",
        "accuracy",
        "balanced_accuracy",
        "brier",
        "log_loss"
    ]
]

,model_name,roc_auc,pr_auc,accuracy,balanced_accuracy,brier,log_loss
0,majority_class_baseline,0.500000,0.430998,0.430998,0.500000,0.569002,20.508899
1,prior_probability_baseline,0.500000,0.430998,0.430998,0.500000,0.265481,0.724373
2,logistic_regression__C_expanded,0.773247,0.705224,0.689592,0.700113,0.201977,0.589763
3,decision_tree__C_expanded,0.772588,0.670857,0.649914,0.675920,0.222864,0.649828
4,random_forest__C_expanded,0.776073,0.715636,0.659990,0.683323,0.219945,0.635572


### Interim note on open final-test performance

Open final-test performance is lower than the Stage 9 trainval apparent metrics and lower than the tuning/CV estimates. This is expected to some extent because Stage 9 trainval metrics are in-sample sanity checks, not generalisation estimates. However, the final test set also has a substantially lower open rate than the train-validation pool (approximately 43.1% vs 57.3%), suggesting chronological campaign-period distribution shift. The models also appear to overestimate open probability on the final test set, especially the random forest. This should be documented later in the final evaluation discussion rather than used to change the models.

## 10.12 Compute click final test metrics

In [40]:
click_metric_rows = []

click_metric_inputs = {
    "majority_class_baseline": click_baseline_probas["majority_class_baseline"],
    "prior_probability_baseline": click_baseline_probas["prior_probability_baseline"],
    "logistic_regression__C_expanded": click_final_test_predictions["proba_logistic_C"].values,
    "random_forest__C_expanded": click_final_test_predictions["proba_random_forest_C"].values,
}

for model_name, y_proba in click_metric_inputs.items():
    metrics = compute_binary_metrics(
        y_true=y_click_test,
        y_proba=y_proba,
        threshold=CLICK_CONTEXTUAL_THRESHOLD
    )
    
    row = {
        "target": "click",
        "model_name": model_name,
        "threshold": CLICK_CONTEXTUAL_THRESHOLD,
        "n_test_rows": len(y_click_test),
        "n_test_positive": int(y_click_test.sum()),
        "test_positive_rate": float(y_click_test.mean()),
        **metrics
    }
    
    click_metric_rows.append(row)

click_final_test_metrics = pd.DataFrame(click_metric_rows)

click_final_test_metrics[
    [
        "model_name",
        "roc_auc",
        "pr_auc",
        "accuracy",
        "balanced_accuracy",
        "brier",
        "log_loss",
        "predicted_positive_count",
        "predicted_positive_rate",
        "proba_min",
        "proba_max",
        "proba_mean"
    ]
]

,model_name,roc_auc,pr_auc,accuracy,balanced_accuracy,brier,log_loss,predicted_positive_count,predicted_positive_rate,proba_min,proba_max,proba_mean
0,majority_class_baseline,0.500000,0.000672,0.999328,0.500000,0.000672,0.024239,0,0.000000,0.000000,0.000000,0.000000
1,prior_probability_baseline,0.500000,0.000672,0.999328,0.500000,0.000672,0.005735,0,0.000000,0.001228,0.001228,0.001228
2,logistic_regression__C_expanded,0.967743,0.126203,0.999190,0.546054,0.000765,0.003649,76,0.000262,0.000007,0.920690,0.001467
3,random_forest__C_expanded,0.970721,0.122027,0.999162,0.553727,0.000848,0.003778,90,0.000310,0.000006,0.812132,0.001639


### Interim observations – click final test

- Both models substantially outperform the dummy baselines.
- Test click prevalence (0.067%) is lower than the train-validation prevalence (~0.123%).
- Logistic regression and random forest achieve very similar final-test PR-AUC values.
- The random forest advantage observed during cross-validation is not clearly preserved on the chronological holdout.
- Further investigation is required through top-k and lift analysis before drawing conclusions regarding model superiority.

## 10.13 Compute top-k/lift analysis

In [41]:
# top-k/lift function
def compute_topk_lift(y_true, y_proba, model_name, topk_fractions):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_proba = pd.Series(y_proba).reset_index(drop=True)

    n = len(y_true)
    total_positives = int(y_true.sum())
    base_rate = y_true.mean()

    ranked = pd.DataFrame({
        "y_true": y_true,
        "y_proba": y_proba
    }).sort_values(
        "y_proba",
        ascending=False
    ).reset_index(drop=True)

    rows = []

    for frac in topk_fractions:
        k = max(1, int(round(n * frac)))
        top_k = ranked.iloc[:k]

        positives_at_k = int(top_k["y_true"].sum())
        precision_at_k = positives_at_k / k
        recall_at_k = positives_at_k / total_positives if total_positives > 0 else np.nan
        lift_at_k = precision_at_k / base_rate if base_rate > 0 else np.nan

        rows.append({
            "model_name": model_name,
            "top_fraction": frac,
            "top_percent": frac * 100,
            "selected_count": k,
            "total_test_rows": n,
            "total_positives": total_positives,
            "base_rate": base_rate,
            "positives_at_k": positives_at_k,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "lift_at_k": lift_at_k,
            "mean_proba_top_k": float(top_k["y_proba"].mean()),
            "min_proba_top_k": float(top_k["y_proba"].min()),
            "max_proba_top_k": float(top_k["y_proba"].max()),
        })

    return pd.DataFrame(rows)

In [42]:
# click top-k/lift
click_topk_lift = pd.concat(
    [
        compute_topk_lift(
            y_true=y_click_test,
            y_proba=click_final_test_predictions["proba_logistic_C"],
            model_name="logistic_regression__C_expanded",
            topk_fractions=TOPK_FRACTIONS
        ),
        compute_topk_lift(
            y_true=y_click_test,
            y_proba=click_final_test_predictions["proba_random_forest_C"],
            model_name="random_forest__C_expanded",
            topk_fractions=TOPK_FRACTIONS
        ),
    ],
    ignore_index=True
)

click_topk_lift

,model_name,top_fraction,top_percent,selected_count,total_test_rows,total_positives,base_rate,positives_at_k,precision_at_k,recall_at_k,lift_at_k,mean_proba_top_k,min_proba_top_k,max_proba_top_k
0,logistic_regression__C_expanded,0.001,0.1,290,289973,195,0.000672,32,0.110345,0.164103,164.087286,0.433715,0.274970,0.920690
1,logistic_regression__C_expanded,0.005,0.5,1450,289973,195,0.000672,103,0.071034,0.528205,105.631190,0.202836,0.072696,0.920690
2,logistic_regression__C_expanded,0.010,1.0,2900,289973,195,0.000672,153,0.052759,0.784615,78.454233,0.120931,0.016555,0.920690
3,logistic_regression__C_expanded,0.050,5.0,14499,289973,195,0.000672,172,0.011863,0.882051,17.640600,0.025367,0.000803,0.920690
4,logistic_regression__C_expanded,0.100,10.0,28997,289973,195,0.000672,179,0.006173,0.917949,9.179582,0.012996,0.000510,0.920690
5,random_forest__C_expanded,0.001,0.1,290,289973,195,0.000672,40,0.137931,0.205128,205.109107,0.475686,0.376121,0.812132
6,random_forest__C_expanded,0.005,0.5,1450,289973,195,0.000672,117,0.080690,0.600000,119.988828,0.259721,0.096804,0.812132
7,random_forest__C_expanded,0.010,1.0,2900,289973,195,0.000672,161,0.055517,0.825641,82.556416,0.148072,0.016283,0.812132
8,random_forest__C_expanded,0.050,5.0,14499,289973,195,0.000672,166,0.011449,0.851282,17.025230,0.031844,0.000571,0.812132
9,random_forest__C_expanded,0.100,10.0,28997,289973,195,0.000672,177,0.006104,0.907692,9.077017,0.016085,0.000205,0.812132


### Interim observations – click top-k analysis

Although logistic regression achieves slightly higher final-test PR-AUC, the random forest consistently outperforms logistic regression in the most selective targeting segments (0.1%, 0.5%, and 1%).

For example, at the top 0.1% segment, the random forest captures 40 clickers compared with 32 for logistic regression, corresponding to a lift of approximately 205x versus 164x.

This suggests that aggregate ranking metrics such as PR-AUC do not fully capture the practical value of the random forest for highly targeted marketing campaigns. The random forest appears particularly effective at identifying the most likely clickers, even though its overall PR-AUC is slightly lower.

In [45]:
# add click top-k to prediction dataframe
def add_topk_flags(pred_df, proba_col, model_label, topk_fractions):
    pred_df = pred_df.copy()
    n = len(pred_df)
    
    ranked_indices = pred_df[proba_col].sort_values(ascending=False).index
    
    for frac in topk_fractions:
        k = max(1, int(round(n * frac)))
        top_indices = ranked_indices[:k]
        
        percent_label = str(frac * 100).replace(".", "_")
        flag_col = f"is_top_{percent_label}pct_{model_label}"
        
        pred_df[flag_col] = 0
        pred_df.loc[top_indices, flag_col] = 1
    
    return pred_df

In [46]:
click_final_test_predictions = add_topk_flags(
    pred_df=click_final_test_predictions,
    proba_col="proba_logistic_C",
    model_label="logistic_C",
    topk_fractions=TOPK_FRACTIONS
)

click_final_test_predictions = add_topk_flags(
    pred_df=click_final_test_predictions,
    proba_col="proba_random_forest_C",
    model_label="random_forest_C",
    topk_fractions=TOPK_FRACTIONS
)

topk_flag_cols = [
    col for col in click_final_test_predictions.columns
    if col.startswith("is_top_")
]

print("Top-k flag columns added:")
for col in topk_flag_cols:
    print(col, "sum:", int(click_final_test_predictions[col].sum()))

Top-k flag columns added:
is_top_0_1pct_logistic_C sum: 290
is_top_0_5pct_logistic_C sum: 1450
is_top_1_0pct_logistic_C sum: 2900
is_top_5_0pct_logistic_C sum: 14499
is_top_10_0pct_logistic_C sum: 28997
is_top_0_1pct_random_forest_C sum: 290
is_top_0_5pct_random_forest_C sum: 1450
is_top_1_0pct_random_forest_C sum: 2900
is_top_5_0pct_random_forest_C sum: 14499
is_top_10_0pct_random_forest_C sum: 28997


In [47]:
click_final_test_predictions[
    click_final_test_predictions["is_top_0_1pct_random_forest_C"] == 1
][
    [
        USER_COL,
        GROUP_COL,
        "y_true",
        "proba_random_forest_C",
        "proba_logistic_C",
        "is_top_0_1pct_random_forest_C",
        "is_top_0_1pct_logistic_C",
    ]
].sort_values(
    "proba_random_forest_C",
    ascending=False
).head(20)

,user_id,mailing_id,y_true,proba_random_forest_C,proba_logistic_C,is_top_0_1pct_random_forest_C,is_top_0_1pct_logistic_C
124860,5c6bed275798a79428359586,1398,1.0,0.812132,0.875504,1,1
124859,5c6bed275798a79428359586,1390,1.0,0.794519,0.907426,1,1
289863,630dbe946168e494f63a11a0,1320,0.0,0.774911,0.635325,1,1
168793,5c6bed455798a7942836941a,1343,0.0,0.769839,0.636599,1,1
124858,5c6bed275798a79428359586,1382,1.0,0.746002,0.872711,1,1
124851,5c6bed275798a79428359586,1288,1.0,0.738660,0.910181,1,1
168797,5c6bed455798a7942836941a,1390,0.0,0.729721,0.618349,1,1
124853,5c6bed275798a79428359586,1301,1.0,0.724346,0.920690,1,1
168790,5c6bed455798a7942836941a,1297,0.0,0.696393,0.688419,1,1
168792,5c6bed455798a7942836941a,1340,0.0,0.684153,0.688267,1,1


In [48]:
topk_flag_summary = []

for col in topk_flag_cols:
    selected = click_final_test_predictions[click_final_test_predictions[col] == 1]
    
    topk_flag_summary.append({
        "flag": col,
        "selected_count": len(selected),
        "actual_clicks_inside": int(selected["y_true"].sum()),
        "precision": selected["y_true"].mean(),
    })

pd.DataFrame(topk_flag_summary)

,flag,selected_count,actual_clicks_inside,precision
0,is_top_0_1pct_logistic_C,290,32,0.110345
1,is_top_0_5pct_logistic_C,1450,103,0.071034
2,is_top_1_0pct_logistic_C,2900,153,0.052759
3,is_top_5_0pct_logistic_C,14499,172,0.011863
4,is_top_10_0pct_logistic_C,28997,179,0.006173
5,is_top_0_1pct_random_forest_C,290,40,0.137931
6,is_top_0_5pct_random_forest_C,1450,117,0.080690
7,is_top_1_0pct_random_forest_C,2900,161,0.055517
8,is_top_5_0pct_random_forest_C,14499,166,0.011449
9,is_top_10_0pct_random_forest_C,28997,177,0.006104


## 10.14 Compare CV vs trainval apparent vs final test

In [49]:
# tuning summaries
open_tuning_summary = pd.read_csv(OPEN_TUNING_SUMMARY_PATH)
click_tuning_summary = pd.read_csv(CLICK_TUNING_SUMMARY_PATH)

print("Open tuning summary shape:", open_tuning_summary.shape)
print("Click tuning summary shape:", click_tuning_summary.shape)

print("\nOpen tuning columns:")
print(open_tuning_summary.columns.tolist())

print("\nClick tuning columns:")
print(click_tuning_summary.columns.tolist())

Open tuning summary shape: (3, 7)
Click tuning summary shape: (2, 7)

Open tuning columns:
['candidate', 'best_params', 'best_score', 'train_score', 'validation_score', 'validation_std', 'train_val_gap']

Click tuning columns:
['candidate', 'best_params', 'best_score', 'train_score', 'validation_score', 'validation_std', 'train_val_gap']


In [51]:
# cv vs trainval vs test

# prepare tuning summaries
open_tuning_comp = open_tuning_summary.copy()
open_tuning_comp["target"] = "open"
open_tuning_comp = open_tuning_comp.rename(columns={
    "candidate": "candidate_key",
    "train_score": "cv_train_primary_metric",
    "validation_score": "cv_validation_primary_metric",
    "validation_std": "cv_validation_std",
    "train_val_gap": "cv_train_val_gap",
})

click_tuning_comp = click_tuning_summary.copy()
click_tuning_comp["target"] = "click"
click_tuning_comp = click_tuning_comp.rename(columns={
    "candidate": "candidate_key",
    "train_score": "cv_train_primary_metric",
    "validation_score": "cv_validation_primary_metric",
    "validation_std": "cv_validation_std",
    "train_val_gap": "cv_train_val_gap",
})

tuning_comp = pd.concat(
    [open_tuning_comp, click_tuning_comp],
    ignore_index=True
)

tuning_comp = tuning_comp[
    [
        "target",
        "candidate_key",
        "cv_train_primary_metric",
        "cv_validation_primary_metric",
        "cv_validation_std",
        "cv_train_val_gap",
    ]
]

# prepare Stage 9 trainval apparent metrics
stage9_comp = stage9_refit_summary.copy()

stage9_comp = stage9_comp[
    [
        "target",
        "candidate_key",
        "trainval_apparent_roc_auc",
        "trainval_apparent_pr_auc",
        "trainval_apparent_brier",
        "trainval_apparent_log_loss",
    ]
]

# prepare Stage 10 final test metrics
open_test_comp = open_final_test_metrics[
    ~open_final_test_metrics["model_name"].str.contains("baseline")
].copy()

open_test_comp = open_test_comp.rename(columns={
    "model_name": "candidate_key",
    "roc_auc": "final_test_roc_auc",
    "pr_auc": "final_test_pr_auc",
    "brier": "final_test_brier",
    "log_loss": "final_test_log_loss",
})

click_test_comp = click_final_test_metrics[
    ~click_final_test_metrics["model_name"].str.contains("baseline")
].copy()

click_test_comp = click_test_comp.rename(columns={
    "model_name": "candidate_key",
    "roc_auc": "final_test_roc_auc",
    "pr_auc": "final_test_pr_auc",
    "brier": "final_test_brier",
    "log_loss": "final_test_log_loss",
})

test_comp = pd.concat(
    [open_test_comp, click_test_comp],
    ignore_index=True
)

test_comp = test_comp[
    [
        "target",
        "candidate_key",
        "final_test_roc_auc",
        "final_test_pr_auc",
        "final_test_brier",
        "final_test_log_loss",
    ]
]

# merge all
cv_vs_test_comparison = (
    tuning_comp
    .merge(stage9_comp, on=["target", "candidate_key"], how="left")
    .merge(test_comp, on=["target", "candidate_key"], how="left")
)

# add target-specific final primary metric
cv_vs_test_comparison["final_test_primary_metric"] = np.where(
    cv_vs_test_comparison["target"] == "open",
    cv_vs_test_comparison["final_test_roc_auc"],
    cv_vs_test_comparison["final_test_pr_auc"]
)

cv_vs_test_comparison["final_vs_cv_validation_gap"] = (
    cv_vs_test_comparison["final_test_primary_metric"]
    - cv_vs_test_comparison["cv_validation_primary_metric"]
)

cv_vs_test_comparison["trainval_apparent_primary_metric"] = np.where(
    cv_vs_test_comparison["target"] == "open",
    cv_vs_test_comparison["trainval_apparent_roc_auc"],
    cv_vs_test_comparison["trainval_apparent_pr_auc"]
)

cv_vs_test_comparison["final_vs_trainval_apparent_gap"] = (
    cv_vs_test_comparison["final_test_primary_metric"]
    - cv_vs_test_comparison["trainval_apparent_primary_metric"]
)

cv_vs_test_comparison[
    [
        "target",
        "candidate_key",
        "cv_train_primary_metric",
        "cv_validation_primary_metric",
        "trainval_apparent_primary_metric",
        "final_test_primary_metric",
        "cv_train_val_gap",
        "final_vs_cv_validation_gap",
        "final_vs_trainval_apparent_gap",
    ]
]

,target,candidate_key,cv_train_primary_metric,cv_validation_primary_metric,trainval_apparent_primary_metric,final_test_primary_metric,cv_train_val_gap,final_vs_cv_validation_gap,final_vs_trainval_apparent_gap
0,open,logistic_regression__C_expanded,0.851958,0.830110,0.850338,0.773247,0.021848,-0.056863,-0.077091
1,open,decision_tree__C_expanded,0.864085,0.828697,0.843003,0.772588,0.035388,-0.056109,-0.070415
2,open,random_forest__C_expanded,0.874846,0.837243,0.869017,0.776073,0.037603,-0.061170,-0.092944
3,click,logistic_regression__C_expanded,0.484600,0.397870,0.478979,0.126203,0.086730,-0.271667,-0.352777
4,click,random_forest__C_expanded,0.919335,0.537474,0.834655,0.122027,0.381860,-0.415447,-0.712628


### Interim observation – CV, trainval apparent, and final test comparison

Final test performance is lower than both CV validation and trainval apparent performance for all selected models. The drop is moderate for open prediction and much larger for click prediction. This suggests that the final chronological holdout represents a harder campaign period, especially for rare-event click prediction.

For open, random forest has the highest final-test ROC-AUC, but logistic regression has better calibration metrics. For click, logistic regression slightly outperforms random forest on aggregate PR-AUC, but the top-k/lift results show that random forest performs better in the most selective targeting segments.

### Investigation Note: Performance Deterioration from Cross-Validation to Final Test

A noticeable decrease in performance is observed when moving from cross-validation estimates to the chronological holdout test set. The magnitude of the decline differs across targets and models, with click prediction showing substantially larger deterioration than open prediction.

At this stage, the underlying cause cannot be determined conclusively. Several plausible explanations exist:

1. Temporal / Campaign-Period Distribution Shift
    * The final test campaigns exhibit substantially lower engagement rates than the train-validation period.
    * Open rate decreases from approximately 57.3% in train-validation to 43.1% in the final test set.
    * Click rate decreases from approximately 0.123% in train-validation to 0.067% in the final test set.
    * These differences suggest that the final test campaigns may represent a different operating environment from the campaigns used during model development.
2. Model Overfitting
    * Some models, particularly the click random forest, exhibit large train-validation gaps during tuning.
    * For example, the click random forest achieves a substantially higher training PR-AUC than validation PR-AUC, indicating that part of the learned structure may not generalize well even within the train-validation period.
3. Combined Effect
    * The observed deterioration may result from both overfitting and distribution shift simultaneously.
    * The available evidence does not currently allow a clean separation between these two mechanisms.

Future Investigation

Potential analyses to perform later include:

* Compare campaign-level engagement rates across train-validation and test periods.
* Examine performance by individual campaign rather than only aggregate performance.
* Analyze calibration drift between train-validation and test periods.
* Investigate whether specific campaign topics, audience segments, or metadata distributions changed over time.
* Review literature on temporal distribution shift, dataset shift, concept drift, covariate shift, and model degradation under changing environments.

At this stage, the performance deterioration should be treated as an open analytical question rather than being attributed solely to either overfitting or distribution shift.

### Literature keywords
- dataset shift machine learning
- distribution shift machine learning
- temporal distribution shift predictive modeling
- concept drift classification
- covariate shift machine learning
- model degradation over time
- temporal validation vs random cross-validation
- out-of-time validation predictive modeling

## 10.15 Save all final-test outputs

In [54]:
FINAL_EVAL_DIR = OUTPUT_DIR / "10_final_evaluation"
FINAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:")
print(FINAL_EVAL_DIR)

Output directory:
/Users/khanhhien/Documents/Thesis/Outputs/10_final_evaluation


In [55]:
# final test predictions
open_predictions_path = (
    FINAL_EVAL_DIR /
    "10_open_final_test_predictions_v1.csv"
)

click_predictions_path = (
    FINAL_EVAL_DIR /
    "10_click_final_test_predictions_v1.csv"
)

open_final_test_predictions.to_csv(
    open_predictions_path,
    index=False
)

click_final_test_predictions.to_csv(
    click_predictions_path,
    index=False
)

print("Saved:")
print(open_predictions_path.name)
print(click_predictions_path.name)

Saved:
10_open_final_test_predictions_v1.csv
10_click_final_test_predictions_v1.csv


In [57]:
# metrics table
open_metrics_path = (
    FINAL_EVAL_DIR /
    "10_open_final_test_metrics_v1.csv"
)

click_metrics_path = (
    FINAL_EVAL_DIR /
    "10_click_final_test_metrics_v1.csv"
)

open_final_test_metrics.to_csv(
    open_metrics_path,
    index=False
)

click_final_test_metrics.to_csv(
    click_metrics_path,
    index=False
)

In [60]:
# top-k/lift
topk_path = (
    FINAL_EVAL_DIR /
    "10_click_topk_lift_v1.csv"
)

click_topk_lift.to_csv(
    topk_path,
    index=False
)

In [58]:
#cv vs test
comparison_path = (
    FINAL_EVAL_DIR /
    "10_cv_vs_test_comparison_v1.csv"
)

cv_vs_test_comparison.to_csv(
    comparison_path,
    index=False
)

In [61]:
# verify
saved_files = sorted(FINAL_EVAL_DIR.glob("*"))

print(f"Files found: {len(saved_files)}\n")

for f in saved_files:
    print(f.name)

Files found: 6

10_click_final_test_metrics_v1.csv
10_click_final_test_predictions_v1.csv
10_click_topk_lift_v1.csv
10_cv_vs_test_comparison_v1.csv
10_open_final_test_metrics_v1.csv
10_open_final_test_predictions_v1.csv


## 10.16 Write final evaluation completion note

In [62]:
completion_note = f"""
Stage 10 final test evaluation completed successfully.

Actions completed:
- Loaded Stage 9 refitted sklearn pipelines.
- Recreated the chronological train-validation/final-test split.
- Verified split consistency, feature columns, and positive-class indices.
- Built final-test dummy baselines using train-validation information only.
- Generated final test predictions for open and click models.
- Computed open final-test metrics.
- Computed click final-test metrics.
- Computed click top-k/lift analysis.
- Compared CV validation, Stage 9 trainval apparent, and Stage 10 final-test performance.
- Saved final-test predictions, metrics, top-k/lift table, and comparison table.

Important:
- No model was retrained in Stage 10.
- No hyperparameters, feature sets, or thresholds were changed after seeing final test results.
- Final test performance should be interpreted as chronological holdout performance.
- Observed performance deterioration may reflect overfitting, distribution shift, or both; this should be discussed later with supporting diagnostics/literature.

Completed at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

COMPLETION_NOTE_PATH = FINAL_EVAL_DIR / "10_final_test_completion_note_v1.txt"

with open(COMPLETION_NOTE_PATH, "w") as f:
    f.write(completion_note)

assert COMPLETION_NOTE_PATH.exists()

print("Stage 10 completion note saved:")
print(COMPLETION_NOTE_PATH)

Stage 10 completion note saved:
/Users/khanhhien/Documents/Thesis/Outputs/10_final_evaluation/10_final_test_completion_note_v1.txt


## 10.17 Final interpretation and findings

### 10.17.1 Overall final-test interpretation

The final chronological test evaluation shows that all selected machine learning models outperform the dummy baselines for both open and click prediction. However, final-test performance is lower than the cross-validation estimates, especially for click prediction. This suggests that the chronological holdout period is more difficult than the campaign periods represented in cross-validation.

The final test results should therefore be interpreted as a realistic future-campaign evaluation rather than as a failure of the modelling pipeline. No post-test retuning or feature changes were performed.

### 10.17.2 Open prediction findings

For open prediction, the three fitted models achieve similar final-test ROC-AUC values, with random forest performing slightly best on ranking metrics.

Final-test ROC-AUC:
- Logistic regression: approximately 0.773
- Decision tree: approximately 0.773
- Random forest: approximately 0.776

The random forest has the highest final-test ROC-AUC and PR-AUC, but logistic regression shows better calibration, with lower Brier score and log loss than the tree-based models. This suggests a trade-off: the random forest ranks users slightly better, while logistic regression produces more reliable probability estimates.

The open final-test event rate is substantially lower than the train-validation event rate. This indicates possible temporal or campaign-period distribution shift, which likely contributes to the decline from cross-validation to final-test performance.

### 10.17.3 Click prediction findings

For click prediction, both logistic regression and random forest substantially outperform dummy baselines. Because click is an extreme rare-event task, PR-AUC and top-k/lift analysis are more informative than accuracy.

Final-test PR-AUC:
- Logistic regression: approximately 0.126
- Random forest: approximately 0.122
- Dummy baseline: approximately 0.00067

On aggregate PR-AUC, logistic regression slightly outperforms random forest. It also has slightly better Brier score and log loss, suggesting better probability calibration.

However, the top-k analysis shows a more nuanced result. The random forest performs better in the most selective targeting segments. At the top 0.1%, the random forest identifies 40 actual clickers compared with 32 for logistic regression. At the top 0.5% and 1%, the random forest also captures more clickers than logistic regression.

This suggests that although logistic regression performs slightly better on aggregate metrics, the random forest may be more useful when the business goal is highly selective targeting of the most likely clickers.

### 10.17.4 Interpretable vs flexible model comparison

The final results do not support a simple conclusion that flexible tree-based models are universally better than logistic regression.

For open prediction, random forest has the strongest ranking performance, but the advantage over logistic regression is small, and logistic regression is better calibrated.

For click prediction, logistic regression slightly outperforms random forest on aggregate PR-AUC and calibration metrics, but random forest performs better in the most selective top-k targeting segments.

Therefore, the answer depends on the modelling objective:
- If the goal is stable, interpretable, and reasonably calibrated prediction, logistic regression remains highly competitive.
- If the goal is to identify a very small group of highest-probability clickers, random forest may offer practical targeting value.
- If the goal is broad final-test generalisation under later campaigns, neither model fully preserves cross-validation performance, especially for click prediction.

### 10.17.5 Performance deterioration: overfitting versus distribution shift

The drop from cross-validation to final test cannot be attributed to a single cause from the current evidence alone. Two mechanisms are plausible.

First, there is evidence of temporal or campaign-period distribution shift. The final test set has lower engagement rates than the train-validation period:
- Open rate decreases from approximately 57.3% to 43.1%.
- Click rate decreases from approximately 0.123% to 0.067%.

Second, there is evidence of overfitting, especially for the click random forest. During tuning, the click random forest showed a large train-validation gap, indicating that it captured patterns that did not fully generalise even within cross-validation.

The most defensible interpretation is that the final-test deterioration reflects a combination of distribution shift and model-specific overfitting. The open models all deteriorate by a similar amount, which points more toward campaign-period shift. The click random forest shows both a large tuning train-validation gap and a large final-test drop, suggesting stronger overfitting risk.

### 10.17.6 Business interpretation

For open prediction, the models can rank likely openers substantially better than dummy baselines, but the campaign-period shift means probability estimates should be interpreted carefully. If calibrated probabilities are important, logistic regression may be preferable. If ranking performance is the priority, random forest has a slight advantage.

For click prediction, aggregate metrics show that both models are useful despite the rarity of clicks. The top-k results are especially relevant for marketing applications. In a highly selective targeting scenario, random forest concentrates clickers more effectively than logistic regression, producing very high lift in the top-ranked segments.

This means the practical value of the click model is not necessarily in predicting every click, but in identifying a small segment of users who are much more likely to click than average.

### 10.17.7 Implications for the research objective

The final evaluation supports a nuanced answer to the research objective. Flexible models can provide additional value, but the value is target- and use-case-dependent.

For open prediction, random forest provides only a small ranking improvement over logistic regression, while logistic regression remains more interpretable and better calibrated.

For click prediction, random forest does not outperform logistic regression on aggregate PR-AUC, but it performs better in the most selective top-k targeting segments. This suggests that flexible models may add value when the objective is highly selective campaign targeting rather than overall probability estimation.

Overall, the results support the importance of evaluating email engagement models using multiple criteria: aggregate discrimination, calibration, rare-event ranking, and business-oriented top-k lift.

### 10.17.8 Note on threshold selection and confusion matrices

For open prediction, a fixed threshold of 0.5 was used to compute contextual classification metrics and confusion-matrix components. This is acceptable because open is a relatively balanced outcome compared with click.

For click prediction, no optimal probability threshold was selected in Stage 10. Since Stage 10 is the final test evaluation, using the final test set to search for an optimal threshold would introduce test-set tuning. In addition, click is an extreme rare-event outcome, so a conventional 0.5 threshold is not practically meaningful. The click evaluation therefore focuses on PR-AUC and pre-specified top-k/lift analysis, which better reflect the ranking and targeting use case.

The confusion-matrix components for click at threshold 0.5 were computed as contextual diagnostics only, not as the primary basis for model evaluation. If a hard click threshold is needed later, it should be selected using validation/CV data or a pre-defined business constraint, then evaluated once on a holdout set.